# Training Visualization with TensorBoard

**TensorBoard** is a toolkit to monitor and visualize model training in real-time — without staring at print statements.

You can track:
- Loss & accuracy curves
- Sample images
- Model predictions
- Model graph structure

In [1]:
!pip install tensorboard

In [2]:
import torch, random, time
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.datasets import FashionMNIST
import torchvision.transforms as T
from torch.utils.tensorboard import SummaryWriter

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else
                       "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

Device: cpu


## 1. Load Data

In [3]:
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

transform = T.Compose([T.ToTensor(), T.Normalize((0.2860,), (0.3530,))])

train_ds = FashionMNIST("./data", train=True,  download=True, transform=transform)
test_ds  = FashionMNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)

100%|██████████| 26.4M/26.4M [00:01<00:00, 16.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 262kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.44MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 7.63MB/s]


## 2. Model

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32*7*7, 128), nn.ReLU(), nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## 3. Initialize TensorBoard SummaryWriter

In [5]:
# Create a unique run folder so previous logs are not overwritten
run_name = f"fashionmnist_{time.strftime('%Y%m%d-%H%M%S')}"
log_dir  = f"./runs/{run_name}"
writer   = SummaryWriter(log_dir=log_dir)
print("Logs will be saved to:", log_dir)
print("\nTo view TensorBoard, run this command in terminal:")
print("  tensorboard --logdir=./runs")

Logs will be saved to: ./runs/fashionmnist_20260517-183849

To view TensorBoard, run this command in terminal:
  tensorboard --logdir=./runs


## 4. Log the Model Graph

In [6]:
# Log the model architecture (can view as graph in TensorBoard → GRAPHS tab)
dummy_input = torch.zeros(1, 1, 28, 28).to(device)
writer.add_graph(model, dummy_input)
print("Model graph logged!")

Model graph logged!


## 5. Train with TensorBoard Logging

In [7]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss   = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

# ── Training loop with TensorBoard ──
epochs = 5
xb_vis, yb_vis = next(iter(test_loader))   # fixed batch for image logging

for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader)
    te_loss, te_acc = evaluate(model, test_loader)

    print(f"Epoch {epoch:02d} | Train Loss: {tr_loss:.4f}, Acc: {tr_acc:.4f} | "
          f"Test Loss: {te_loss:.4f}, Acc: {te_acc:.4f}")

    # ── Log scalars (Loss / Accuracy curves) ──
    writer.add_scalar("Loss/train",      tr_loss, epoch)
    writer.add_scalar("Loss/test",       te_loss, epoch)
    writer.add_scalar("Accuracy/train",  tr_acc,  epoch)
    writer.add_scalar("Accuracy/test",   te_acc,  epoch)

    # ── Log sample test images ──
    writer.add_images("Images/test_samples", xb_vis[:16], epoch)

    # ── Log predictions vs true labels ──
    model.eval()
    with torch.no_grad():
        preds = model(xb_vis.to(device)).argmax(1).cpu()

    pred_text = "\n".join([
        f"i={i}: pred={class_names[int(preds[i])]}, true={class_names[int(yb_vis[i])]}"
        for i in range(16)
    ])
    writer.add_text("Predictions/sample_batch", pred_text, epoch)

writer.close()
print("\nDone! Run: tensorboard --logdir=./runs")

Epoch 01 | Train Loss: 0.4568, Acc: 0.8349 | Test Loss: 0.3554, Acc: 0.8702
Epoch 02 | Train Loss: 0.3010, Acc: 0.8913 | Test Loss: 0.3218, Acc: 0.8835
Epoch 03 | Train Loss: 0.2572, Acc: 0.9051 | Test Loss: 0.2698, Acc: 0.8992
Epoch 04 | Train Loss: 0.2265, Acc: 0.9170 | Test Loss: 0.2745, Acc: 0.9013
Epoch 05 | Train Loss: 0.2030, Acc: 0.9245 | Test Loss: 0.2603, Acc: 0.9061

Done! Run: tensorboard --logdir=./runs


## 6. TensorBoard Tabs — What to Look For
| Tab | What you see |
|---|---|
| **SCALARS** | Loss & accuracy curves per epoch |
| **IMAGES** | Sample test images per epoch |
| **TEXT** | Prediction logs |
| **GRAPHS** | Model architecture |
| **HISTOGRAMS** | Weight distributions (if added) |

```bash
# Launch in terminal:
tensorboard --logdir=./runs
# Then open:  http://localhost:6006
```

## 7. Logging Histograms (Bonus)

In [8]:
# Reopen writer to add histogram logging example
writer2 = SummaryWriter(log_dir=log_dir + "_hist")

# Simulate one epoch with histogram logging
model2 = SimpleCNN().to(device)
opt2   = optim.Adam(model2.parameters(), lr=1e-3)

for epoch in range(1, 4):
    model2.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt2.zero_grad()
        loss = criterion(model2(xb), yb)
        loss.backward(); opt2.step()

    # Log weight histograms
    for name, param in model2.named_parameters():
        writer2.add_histogram(f"Weights/{name}", param.data, epoch)
        if param.grad is not None:
            writer2.add_histogram(f"Gradients/{name}", param.grad, epoch)

    writer2.add_scalar("Loss/train", loss.item(), epoch)

writer2.close()
print("Histogram logs saved!")

Histogram logs saved!
